In [ ]:
# Copyright 2026 EY. All rights reserved
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
#
# SPDX-License-Identifier: Apache-2.0

# Model ProvenanceKit —  Benchmark

Run the **ProvenanceKit** provenance pipeline on the ** Benchmark** dataset and evaluate results.

The benchmark tests whether `provenancekit compare` can correctly distinguish
**similar** model pairs (identity, fine-tuning, distillation, quantization) from
**dissimilar** pairs (independently trained models).

## Signals
MFI · TFV · VOA · EAS · NLF · LEP · END · WVC

## Scoring
MFI-gated identity score (5-signal weighted average)

## Configuration
| Parameter | Default | Description |
|-----------|---------|-------------|
| `MAX_WORKERS` | 2 | Parallel comparison workers (increase for faster runs on multi-core machines) |
| `PAIR_LIMIT` | `None` | Max pairs to evaluate (`None` = all). Useful for quick smoke-tests |
| `PAIR_FILTER` | `"all"` | Which pairs to run: `"all"`, `"similar"`, or `"dissimilar"` |

In [ ]:
import json
import os
import threading
import time
from concurrent.futures import FIRST_COMPLETED, ThreadPoolExecutor, wait
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import HTML, display

from provenancekit.config.settings import Settings
from provenancekit.core.scanner import ModelProvenanceScanner
from provenancekit.services.cache import CacheService

## Configuration

Tweak these parameters before running the benchmark.
- **`MAX_WORKERS`** — number of parallel workers (default: 2; increase to speed up on multi-core machines).
- **`PAIR_LIMIT`** — set to an integer to run only a subset of pairs (e.g. `10` for a quick test), or `None` for all.
- **`PAIR_FILTER`** — `"all"` (default), `"similar"`, or `"dissimilar"` to benchmark only one category.

In [ ]:
BENCHMARK_FILE = Path("Benchmark_All.json")

# ── Parallelism ──────────────────────────────────────────────────────
# Default: 2 workers (safe for most machines).  Increase for faster runs.
MAX_WORKERS = 2

# ── Pair filtering ───────────────────────────────────────────────────
# PAIR_FILTER: "all" | "similar" | "dissimilar"
PAIR_FILTER = "similar"
# PAIR_LIMIT: int or None — max number of pairs to evaluate after filtering
PAIR_LIMIT = 5  # e.g. set to 10 for a quick smoke-test

VERBOSE = True

# ── Optional overrides (None = use defaults) ─────────────────────────
CACHE_DIR = None     # Path to custom cache directory
DB_ROOT = None       # Path to custom provenance database root

print(f"Benchmark file : {BENCHMARK_FILE}")
print(f"Max workers    : {MAX_WORKERS}  (CPUs available: {os.cpu_count()})")
print(f"Pair filter    : {PAIR_FILTER}")
print(f"Pair limit     : {PAIR_LIMIT or 'all'}")

## Parse & Filter Benchmark JSON

Loads the benchmark file and applies `PAIR_FILTER` / `PAIR_LIMIT`.

In [ ]:
def parse_benchmark(benchmark_file):
    """Load benchmark JSON and return (flat_list, raw_dict)."""
    with Path(benchmark_file).open() as f:
        bench = json.load(f)

    all_pairs = []
    for entry in bench.get("similar", []):
        if "pair" not in entry:
            continue
        all_pairs.append({
            "pair": entry["pair"],
            "label": "similar",
            "arch_types": entry.get("type", []),
            "challenge": entry.get("challenge", ""),
            "difficulty": entry.get("difficulty", ""),
            "constitution": entry.get("provenance", ""),
        })
    for entry in bench.get("dissimilar", []):
        if "pair" not in entry:
            continue
        all_pairs.append({
            "pair": entry["pair"],
            "label": "dissimilar",
            "arch_types": entry.get("type", []),
            "challenge": entry.get("challenge", ""),
            "difficulty": entry.get("difficulty", ""),
            "constitution": entry.get("confusion_type", ""),
        })
    return all_pairs, bench


all_pairs_raw, bench_raw = parse_benchmark(BENCHMARK_FILE)
n_sim_total = sum(1 for p in all_pairs_raw if p["label"] == "similar")
n_dis_total = sum(1 for p in all_pairs_raw if p["label"] == "dissimilar")
print(
    f"Loaded {len(all_pairs_raw)} pairs "
    f"({n_sim_total} similar + {n_dis_total} dissimilar)"
)

# Apply PAIR_FILTER
if PAIR_FILTER == "similar":
    all_pairs = [p for p in all_pairs_raw if p["label"] == "similar"]
elif PAIR_FILTER == "dissimilar":
    all_pairs = [p for p in all_pairs_raw if p["label"] == "dissimilar"]
else:
    all_pairs = list(all_pairs_raw)

# Apply PAIR_LIMIT
if PAIR_LIMIT is not None and PAIR_LIMIT > 0:
    all_pairs = all_pairs[:PAIR_LIMIT]

n_sim = sum(1 for p in all_pairs if p["label"] == "similar")
n_dis = sum(1 for p in all_pairs if p["label"] == "dissimilar")
unique_models = sorted({m for item in all_pairs for m in item["pair"]})

print(f"After filtering: {len(all_pairs)} pairs ({n_sim} similar + {n_dis} dissimilar)")
print(f"Unique models: {len(unique_models)}")

## Helper: Flatten ProvenanceKit CompareResult to Dict

In [ ]:
def flatten_result(pair_item, result):
    """Convert a CompareResult + pair metadata into a flat dict for DataFrame."""
    id_a, id_b = pair_item["pair"]
    return {
        "model_a": id_a,
        "model_b": id_b,
        "short_a": id_a.split("/")[-1],
        "short_b": id_b.split("/")[-1],
        "label": pair_item["label"],
        "difficulty": pair_item.get("difficulty", ""),
        "challenge": pair_item.get("challenge", ""),
        "constitution": pair_item.get("constitution", ""),
        "family_a": result.family_a,
        "family_b": result.family_b,
        "mfi_score": result.scores.mfi_score,
        "mfi_tier": result.scores.mfi_tier,
        "mfi_match": result.scores.mfi_match,
        "tfv": result.signals.tfv,
        "voa": result.signals.voa,
        "eas": result.signals.eas,
        "nlf": result.signals.nlf,
        "lep": result.signals.lep,
        "end": result.signals.end,
        "wvc": result.signals.wvc,
        "tokenizer_score": result.scores.tokenizer_score,
        "identity_score": result.scores.identity_score,
        "pipeline_score": result.scores.pipeline_score,
        "provenance_decision": result.scores.provenance_decision,
        "time": result.time_seconds,
    }


def make_error_row(pair_item, error_msg):
    """Return a row with NaN scores for a failed pair."""
    id_a, id_b = pair_item["pair"]
    return {
        "model_a": id_a,
        "model_b": id_b,
        "short_a": id_a.split("/")[-1],
        "short_b": id_b.split("/")[-1],
        "label": pair_item["label"],
        "difficulty": pair_item.get("difficulty", ""),
        "challenge": pair_item.get("challenge", ""),
        "constitution": pair_item.get("constitution", ""),
        "family_a": "error",
        "family_b": "error",
        "mfi_score": np.nan,
        "mfi_tier": 0,
        "mfi_match": "error",
        "tfv": np.nan,
        "voa": np.nan,
        "eas": np.nan,
        "nlf": np.nan,
        "lep": np.nan,
        "end": np.nan,
        "wvc": np.nan,
        "tokenizer_score": np.nan,
        "identity_score": np.nan,
        "pipeline_score": np.nan,
        "provenance_decision": "error",
        "time": 0,
    }

## Run Benchmark

This cell runs `provenancekit compare` (via `ModelProvenanceScanner.compare()`) for every selected pair in parallel.
Depending on the number of pairs and model sizes, this can take a while.

In [ ]:
_print_lock = threading.Lock()

def _tprint(msg):
    """Thread-safe print."""
    with _print_lock:
        print(msg, flush=True)


TIMEOUT_PER_PAIR = 600  # 10 min per pair

# Build scanner
settings_kwargs = {}
if CACHE_DIR is not None:
    settings_kwargs["cache_dir"] = Path(CACHE_DIR)
if DB_ROOT is not None:
    settings_kwargs["db_root"] = Path(DB_ROOT)
settings = Settings(**settings_kwargs)
cache = CacheService(cache_dir=settings.cache_dir)
scanner = ModelProvenanceScanner(settings=settings, cache=cache)

n_pairs = len(all_pairs)

print("=" * 80)
print("  ProvenanceKit — Benchmark")
print(f"  Pairs: {n_pairs} ({n_sim} similar + {n_dis} dissimilar)")
print(f"  Workers: {MAX_WORKERS}  |  CPUs: {os.cpu_count()}")
print("=" * 80)


def _eval_pair(idx_item):
    idx, item = idx_item
    id_a, id_b = item["pair"]
    try:
        result = scanner.compare(id_a, id_b)
        return idx, flatten_result(item, result), None
    except Exception as e:
        return idx, make_error_row(item, str(e)), str(e)


t_start = time.time()
results = [None] * n_pairs

if MAX_WORKERS > 1:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {
            pool.submit(_eval_pair, (i, item)): i
            for i, item in enumerate(all_pairs)
        }
        pending = set(futures.keys())
        done_count = 0
        stalled = []

        while pending:
            done, pending = wait(
                pending, timeout=TIMEOUT_PER_PAIR, return_when=FIRST_COMPLETED,
            )
            if not done:
                for fut in list(pending):
                    orig_idx = futures[fut]
                    fut.cancel()
                    stalled.append((orig_idx, all_pairs[orig_idx]))
                    done_count += 1
                    pair = all_pairs[orig_idx]["pair"]
                    if VERBOSE:
                        _tprint(
                            f"  [{done_count:3d}/{n_pairs}] "
                            f"{pair[0].split('/')[-1]} vs {pair[1].split('/')[-1]}"
                            f"  STALLED (>{TIMEOUT_PER_PAIR}s)"
                        )
                pending = set()
                break

            for fut in done:
                orig_idx = futures[fut]
                try:
                    idx, row, err = fut.result()
                    results[idx] = row
                    done_count += 1
                    if VERBOSE:
                        if err:
                            _tprint(
                                f"  [{done_count:3d}/{n_pairs}] "
                                f"{row['short_a']} vs {row['short_b']:<25s} "
                                f"ERROR: {err}"
                            )
                        else:
                            _tprint(
                                f"  [{done_count:3d}/{n_pairs}] "
                                f"{row['short_a']} vs {row['short_b']:<25s} "
                                f"Pipeline={row['pipeline_score']:.3f}  "
                                f"{row['time']:.1f}s"
                            )
                except Exception as e:
                    done_count += 1
                    pair = all_pairs[orig_idx]["pair"]
                    if VERBOSE:
                        _tprint(
                            f"  [{done_count:3d}/{n_pairs}] "
                            f"{pair[0].split('/')[-1]} vs {pair[1].split('/')[-1]}"
                            f"  FAILED: {e}"
                        )

        # Retry stalled pairs sequentially
        if stalled:
            _tprint(f"\n  Retrying {len(stalled)} stalled pair(s) sequentially ...")
            for idx, item in stalled:
                try:
                    _, row, err = _eval_pair((idx, item))
                    results[idx] = row
                    status = "OK" if not err else f"ERROR: {err}"
                    if VERBOSE:
                        _tprint(
                            f"  [retry] {row['short_a']} vs {row['short_b']:<25s} "
                            f"{status}"
                        )
                except Exception as e:
                    pair = item["pair"]
                    if VERBOSE:
                        _tprint(
                            f"  [retry] {pair[0].split('/')[-1]} vs "
                            f"{pair[1].split('/')[-1]:<25s} FAILED: {e}"
                        )
else:
    for idx, item in enumerate(all_pairs):
        _, row, err = _eval_pair((idx, item))
        results[idx] = row
        if VERBOSE:
            if err:
                _tprint(
                    f"  [{idx+1:3d}/{n_pairs}] "
                    f"{row['short_a']} vs {row['short_b']:<25s} ERROR: {err}"
                )
            else:
                _tprint(
                    f"  [{idx+1:3d}/{n_pairs}] "
                    f"{row['short_a']} vs {row['short_b']:<25s} "
                    f"Pipeline={row['pipeline_score']:.3f}  {row['time']:.1f}s"
                )

t_done = time.time()

# Fill None entries (stalled pairs that were never retried)
for i in range(n_pairs):
    if results[i] is None:
        results[i] = make_error_row(all_pairs[i], "stalled/cancelled")

print(f"\n{'=' * 80}")
print(f"Benchmark Complete: {n_pairs} pairs in {t_done - t_start:.1f}s")
print(f"{'=' * 80}")

# Build DataFrames
df_bench = pd.DataFrame(results)

display_cols = [
    "short_a", "short_b", "label", "constitution", "difficulty", "challenge",
    "mfi_score", "mfi_tier", "mfi_match",
    "tfv", "voa",
    "eas", "nlf", "lep", "end", "wvc",
    "tokenizer_score", "identity_score",
    "pipeline_score",
]
display_names = [
    "Model A", "Model B", "Label", "Constitution", "Difficulty", "Challenge",
    "MFI", "Tier", "Match",
    "TFV", "VOA",
    "EAS", "NLF", "LEP", "END", "WVC",
    "TokScore", "IdScore",
    "Pipeline",
]

df_display = df_bench[display_cols].copy()
df_display.columns = display_names

## Display Results

Colour-coded table, MFI tier distribution, score distributions, and box-plots.

In [ ]:
gradient_cols = {"MFI", "Pipeline", "TokScore", "IdScore"}


def _cell_styles(row):
    styles = []
    for col in row.index:
        if col == "Label":
            if row["Label"] == "similar":
                styles.append(
                    "background-color: #28a745; color: white; font-weight: bold"
                )
            else:
                styles.append(
                    "background-color: #dc3545; color: white; font-weight: bold"
                )
        elif col in gradient_cols:
            styles.append("color: #111")
        else:
            styles.append("color: white")
    return styles


display(HTML("<h3>ProvenanceKit Results \u2014 MFI-gated Identity Score</h3>"))
styler = (
    df_display.style
    .apply(_cell_styles, axis=1)
    .format(precision=4)
    .background_gradient(subset=["Pipeline"], cmap="RdYlGn", vmin=0, vmax=1)
    .background_gradient(subset=["MFI"], cmap="RdYlGn", vmin=0, vmax=1)
    .background_gradient(subset=["TokScore"], cmap="RdYlGn", vmin=0, vmax=1)
    .background_gradient(subset=["IdScore"], cmap="RdYlGn", vmin=0, vmax=1)
    .set_caption("Benchmark \u2014 Pipeline Score (MFI-gated Identity)")
    .set_table_styles([
        {"selector": "caption",
         "props": [("font-size", "16px"), ("font-weight", "bold"),
                   ("color", "white")]},
        {"selector": "td",
         "props": [("font-size", "11px")]},
        {"selector": "th",
         "props": [("color", "white"), ("font-weight", "bold"),
                   ("font-size", "11px")]},
    ])
)
if "Challenge" in df_display.columns:
    styler = styler.set_properties(
        subset=["Challenge"],
        **{"max-width": "250px", "white-space": "normal",
           "text-align": "left", "font-size": "10px"})
if "Difficulty" in df_display.columns:
    styler = styler.set_properties(
        subset=["Difficulty"],
        **{"text-align": "center", "font-size": "10px"})
display(styler)

# MFI Tier distribution
print("\n" + "=" * 70)
print("  MFI TIER DISTRIBUTION")
print("=" * 70)
for label in ["similar", "dissimilar"]:
    subset = df_bench[df_bench["label"] == label].dropna(subset=["mfi_tier"])
    tier_counts = subset["mfi_tier"].value_counts().sort_index()
    print(f"\n  {label.upper()} pairs:")
    for t in [1, 2, 3]:
        n = tier_counts.get(t, 0)
        pct = n / len(subset) * 100 if len(subset) > 0 else 0
        print(f"    Tier {t}: {n:2d} ({pct:.0f}%)")

# Score distributions
print("\n" + "=" * 70)
print("  SCORE DISTRIBUTIONS BY LABEL")
print("=" * 70)
score_cols = [
    ("pipeline_score", "Pipeline"),
    ("mfi_score", "MFI"),
    ("tokenizer_score", "TokScore"), ("identity_score", "IdScore"),
    ("tfv", "TFV"), ("voa", "VOA"),
    ("eas", "EAS"), ("nlf", "NLF"), ("lep", "LEP"),
    ("end", "END"), ("wvc", "WVC"),
]
for label in ["similar", "dissimilar"]:
    valid = df_bench[df_bench["label"] == label].dropna(subset=["pipeline_score"])
    if len(valid) == 0:
        continue
    print(f"\n  {label.upper()} pairs (n={len(valid)}):")
    for col, name in score_cols:
        if col in valid.columns:
            vals = valid[col]
            print(f"    {name:12s}  mean={vals.mean():.4f}  std={vals.std():.4f}"
                  f"  min={vals.min():.4f}  max={vals.max():.4f}")

# Box-plots
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
box_configs = [
    ("identity_score", "Identity Score (EAS+NLF+LEP+END+WVC)"),
    ("tokenizer_score", "Tokenizer Score (reporting only)"),
    ("pipeline_score", "Pipeline Score (MFI-gated Identity)"),
]
for i, (col, title) in enumerate(box_configs):
    ax = axes[i]
    sim_vals = df_bench[df_bench["label"] == "similar"][col].dropna()
    dis_vals = df_bench[df_bench["label"] == "dissimilar"][col].dropna()
    bp = ax.boxplot(
        [sim_vals, dis_vals], labels=["Similar", "Dissimilar"],
        patch_artist=True, widths=0.6,
    )
    bp["boxes"][0].set_facecolor("#d4edda")
    bp["boxes"][1].set_facecolor("#f8d7da")
    for j, (vals, x) in enumerate([(sim_vals, 1), (dis_vals, 2)]):
        ax.scatter(
            np.full(len(vals), x) + np.random.normal(0, 0.04, len(vals)),
            vals, alpha=0.5, s=30, c=["#2ecc71", "#e74c3c"][j], zorder=3,
        )
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_ylabel("Score")
    ax.set_ylim(-0.05, 1.1)
    ax.grid(True, alpha=0.3)

plt.suptitle("Score Separation: Similar vs Dissimilar Pairs",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## Classification Analysis

Sweep thresholds on the results to find the best F1, show confusion matrix and metrics.

In [ ]:
def sweep_threshold(scores, y_true, label=""):
    """Sweep thresholds and return (metrics_df, best_threshold, best_f1)."""
    thresholds = np.arange(0.15, 0.90, 0.01)
    best_f1, best_t = 0, 0.5
    rows = []
    for t in thresholds:
        y_pred = (scores >= t).astype(int)
        tp = ((y_pred == 1) & (y_true == 1)).sum()
        fp = ((y_pred == 1) & (y_true == 0)).sum()
        tn = ((y_pred == 0) & (y_true == 0)).sum()
        fn = ((y_pred == 0) & (y_true == 1)).sum()
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
        acc = (tp + tn) / len(y_true)
        rows.append({
            "threshold": round(t, 2), "accuracy": acc,
            "precision": prec, "recall": rec, "f1": f1,
            "tp": tp, "fp": fp, "tn": tn, "fn": fn,
        })
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return pd.DataFrame(rows), best_t, best_f1


# Run analysis
valid = df_bench.dropna(subset=["pipeline_score"]).copy()
y_true = (valid["label"] == "similar").astype(int).values

continuous = [
    ("Pipeline", "pipeline_score"),
    ("IdScore", "identity_score"),
    ("TokScore", "tokenizer_score"),
]

best_results = {}
for name, col in continuous:
    if col not in valid.columns:
        continue
    scores = valid[col].fillna(0).values
    df_m, bt, bf = sweep_threshold(scores, y_true, label=name)
    best_results[name] = {"df": df_m, "best_t": bt, "best_f1": bf}

# F1 vs Threshold plot
plot_methods = [m for m in ["Pipeline", "IdScore", "TokScore"] if m in best_results]
colors = {
    "Pipeline": "#2c3e50",
    "IdScore": "#2980b9",
    "TokScore": "#e67e22",
}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for method in plot_methods:
    br = best_results[method]
    axes[0].plot(
        br["df"]["threshold"], br["df"]["f1"], "-",
        label=f"{method} (F1={br['best_f1']:.3f} @ {br['best_t']:.2f})",
        color=colors.get(method, "#333"), linewidth=2,
    )
    axes[0].axvline(
        br["best_t"], color=colors.get(method, "#333"),
        linestyle=":", alpha=0.4,
    )

axes[0].set_xlabel("Threshold")
axes[0].set_ylabel("F1 Score")
axes[0].set_title("F1 vs Threshold", fontsize=12, fontweight="bold")
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, 1.05)

# Confusion matrix for Pipeline Score
if "Pipeline" in best_results:
    br = best_results["Pipeline"]
    best_row = br["df"].loc[
        (br["df"]["threshold"] - br["best_t"]).abs().idxmin()
    ]
    cm = np.array([
        [int(best_row["tn"]), int(best_row["fp"])],
        [int(best_row["fn"]), int(best_row["tp"])],
    ])
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues", ax=axes[1],
        xticklabels=["Pred: Dissimilar", "Pred: Similar"],
        yticklabels=["True: Dissimilar", "True: Similar"],
        square=True, linewidths=1, cbar=False,
        annot_kws={"size": 18, "weight": "bold"},
    )
    axes[1].set_title(
        f"Pipeline Score\n(t={br['best_t']:.2f}, F1={br['best_f1']:.3f})",
        fontsize=11, fontweight="bold",
    )

plt.tight_layout()
plt.show()

# Summary table
print("\n" + "=" * 70)
print("  ProvenanceKit — BEST METRICS: PIPELINE SCORE")
print("=" * 70)
header = (
    f"  {'Method':<12s} {'Thresh':>7s} {'Acc':>7s} "
    f"{'Prec':>7s} {'Rec':>7s} {'F1':>7s}  TP  FP  TN  FN"
)
print(f"\n{header}")
print("  " + "-" * 75)

for method in ["Pipeline", "IdScore", "TokScore"]:
    if method not in best_results:
        continue
    br = best_results[method]
    row = br["df"].loc[
        (br["df"]["threshold"] - br["best_t"]).abs().idxmin()
    ]
    t_str = f"{br['best_t']:.2f}"
    print(
        f"  {method:<12s} {t_str:>7s} {row['accuracy']:>7.3f}"
        f" {row['precision']:>7.3f} {row['recall']:>7.3f}"
        f" {row['f1']:>7.3f}  {int(row['tp']):2d}  {int(row['fp']):2d}"
        f"  {int(row['tn']):2d}  {int(row['fn']):2d}"
    )

best_threshold = best_results.get("Pipeline", {}).get("best_t", 0.5)
best_f1 = best_results.get("Pipeline", {}).get("best_f1", 0)

print(f"\n  -> Pipeline Score: threshold {best_threshold:.2f}, F1 {best_f1:.3f}")

## Error Analysis

Inspect misclassified pairs from the benchmark, visualize per-pair scores and signal distributions.

In [ ]:
threshold = best_threshold

valid = df_bench.dropna(subset=["pipeline_score"]).copy()
valid["pred"] = np.where(
    valid["pipeline_score"] >= threshold, "similar", "dissimilar",
)
valid["correct"] = valid["label"] == valid["pred"]

errors = valid[~valid["correct"]].copy()
print(f"\n{'=' * 80}")
print(
    f"  MISCLASSIFICATIONS \u2014 Pipeline Score "
    f"({len(errors)} errors at threshold {threshold:.2f})"
)
print(f"{'=' * 80}")

if len(errors) == 0:
    print("  No misclassified pairs \u2014 perfect classification!")
else:
    for _, row in errors.iterrows():
        etype = "FALSE POSITIVE" if row["pred"] == "similar" else "FALSE NEGATIVE"
        print(f"\n  [{etype}] {row['short_a']} vs {row['short_b']}")
        print(f"    True: {row['label']}  |  Predicted: {row['pred']}")
        print(
            f"    MFI={row['mfi_score']:.4f}(T{row['mfi_tier']})  "
            f"TFV={row['tfv']:.4f}  VOA={row['voa']:.4f}"
        )
        print(
            f"    EAS={row['eas']:.4f}  NLF={row['nlf']:.4f}  "
            f"LEP={row['lep']:.4f}  END={row['end']:.4f}  "
            f"WVC={row['wvc']:.4f}"
        )
        print(
            f"    TokScore={row['tokenizer_score']:.4f}  "
            f"IdScore={row['identity_score']:.4f}  "
            f"Pipeline={row['pipeline_score']:.4f}"
        )
        print(f"    Families: {row['family_a']} vs {row['family_b']}")
        print(f"    Challenge: {row['challenge']}")

# Scatter plot
fig, ax = plt.subplots(1, 1, figsize=(14, 6))

sim_correct = valid[(valid["label"] == "similar") & valid["correct"]]
dis_correct = valid[(valid["label"] == "dissimilar") & valid["correct"]]
sim_err = errors[errors["label"] == "similar"]
dis_err = errors[errors["label"] == "dissimilar"]

ax.scatter(
    range(len(sim_correct)), sim_correct["pipeline_score"].values,
    c="#2ecc71", s=80, label="Similar (correct)", marker="o", zorder=3,
)
offset = len(sim_correct)
for _, row in sim_err.iterrows():
    ax.scatter(
        offset, row["pipeline_score"], c="#e74c3c", s=120,
        marker="X", zorder=4, edgecolors="black", linewidths=1,
    )
    ax.annotate(
        f"{row['short_a'][:10]}\nvs {row['short_b'][:10]}",
        (offset, row["pipeline_score"]),
        textcoords="offset points", xytext=(8, 5),
        fontsize=7, color="#e74c3c",
    )
    offset += 1
offset2 = offset
ax.scatter(
    range(offset2, offset2 + len(dis_correct)),
    dis_correct["pipeline_score"].values,
    c="#e74c3c", s=80, label="Dissimilar (correct)", marker="s", zorder=3,
)
offset2 += len(dis_correct)
for _, row in dis_err.iterrows():
    ax.scatter(
        offset2, row["pipeline_score"], c="#2ecc71", s=120,
        marker="X", zorder=4, edgecolors="black", linewidths=1,
    )
    ax.annotate(
        f"{row['short_a'][:10]}\nvs {row['short_b'][:10]}",
        (offset2, row["pipeline_score"]),
        textcoords="offset points", xytext=(8, 5),
        fontsize=7, color="#2ecc71",
    )
    offset2 += 1

ax.axhline(
    threshold, color="gray", linestyle="--", alpha=0.7,
    label=f"Threshold ({threshold:.2f})",
)
n_err = len(errors)
ax.set_xlabel("Pair Index (similar \u2192 dissimilar)")
ax.set_ylabel("Pipeline Score")
ax.set_title(
    f"Pipeline Score ({n_err} errors)",
    fontsize=12, fontweight="bold",
)
ax.legend(loc="center right", fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle(
    "Pipeline: Per-Pair Scores (X = misclassified)",
    fontsize=14, fontweight="bold", y=1.02,
)
plt.tight_layout()
plt.show()

# Mean signal bars
fig, ax = plt.subplots(1, 1, figsize=(14, 6))
signals = [
    "mfi_score", "tfv", "voa", "eas", "nlf", "lep", "end", "wvc",
    "tokenizer_score", "identity_score", "pipeline_score",
]
signal_names = [
    "MFI", "TFV", "VOA", "EAS", "NLF", "LEP", "END", "WVC",
    "TokS", "IdS", "Pipeline",
]
sim_means = [valid[valid["label"] == "similar"][s].mean() for s in signals]
dis_means = [valid[valid["label"] == "dissimilar"][s].mean() for s in signals]

x = np.arange(len(signals))
w = 0.35
ax.bar(
    x - w / 2, sim_means, w, label="Similar (mean)",
    color="#2ecc71", edgecolor="white",
)
ax.bar(
    x + w / 2, dis_means, w, label="Dissimilar (mean)",
    color="#e74c3c", edgecolor="white",
)
ax.set_xticks(x)
ax.set_xticklabels(signal_names, rotation=45, ha="right")
ax.set_ylabel("Mean Score")
ax.set_title("Mean Signal Scores by Label", fontsize=12, fontweight="bold")
ax.legend()
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3, axis="y")

for i in range(len(signals)):
    gap = sim_means[i] - dis_means[i]
    y_pos = max(sim_means[i], dis_means[i]) + 0.03
    ax.annotate(
        f"{gap:+.2f}", (x[i], y_pos), ha="center", fontsize=7,
        color="#2ecc71" if gap > 0 else "#e74c3c", fontweight="bold",
    )

plt.tight_layout()
plt.show()

# Per-signal box-plots
fig, axes = plt.subplots(2, 6, figsize=(30, 10))
all_signals = [
    ("mfi_score", "MFI"), ("tfv", "TFV"), ("voa", "VOA"),
    ("eas", "EAS"), ("nlf", "NLF"), ("lep", "LEP"),
    ("end", "END"), ("wvc", "WVC"),
    ("tokenizer_score", "TokScore"), ("identity_score", "IdScore"),
    ("pipeline_score", "Pipeline"),
]
for i, (signal, name) in enumerate(all_signals):
    row_idx = i // 6
    col_idx = i % 6
    ax = axes[row_idx][col_idx]
    sim_vals = valid[valid["label"] == "similar"][signal].dropna()
    dis_vals = valid[valid["label"] == "dissimilar"][signal].dropna()
    bp = ax.boxplot(
        [sim_vals, dis_vals], labels=["Sim", "Dis"],
        patch_artist=True, widths=0.6,
    )
    bp["boxes"][0].set_facecolor("#d4edda")
    bp["boxes"][1].set_facecolor("#f8d7da")
    ax.set_title(name, fontsize=11, fontweight="bold")
    ax.set_ylabel("Score" if col_idx == 0 else "")
    ax.set_ylim(-0.05, 1.1)
    ax.grid(True, alpha=0.3)

axes[1][5].set_visible(False)

plt.suptitle(
    "Signal Distributions: Similar vs Dissimilar",
    fontsize=14, fontweight="bold",
)
plt.tight_layout()
plt.show()